# 25｜附錄B_混合精度與量化

執行對稱 int8 量化，直接比較原權重、整數碼與反量化誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/25_附錄B_混合精度與量化.md`，目前 28 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

浮點位元/BF16 模擬、loss scaling、量化粒度/PTQ/QAT/STE。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 25-03 | 12、14、16 | 課堂小實驗 |
| 25-04 | 3、14、16 | 執行結果圖 |
| 25-05 | 4、5、6、7、8、9、10、11、25、28 | 1. 位元格式、BF16 模擬與 loss scaling |
| 25-06 | 12、13、14、15、16、18、25、26、28 | 2. 手算、零張量、對稱／非對稱與每通道 |
| 25-07 | 17、18、19、20、21、22、23、24、25、26、27、28 | 3. 校準、動態範圍與 fake quantization |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 25-01｜投影片 25：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 25-02｜投影片 25：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

In [ ]:
# 程式 25-03｜投影片 25：12、14、16 頁
weights = np.array([[-1.20, -0.35, 0.0, 0.42, 1.05], [0.18, -0.77, 0.63, 0.91, -0.12]])
scale = np.max(np.abs(weights)) / 127
q = np.clip(np.round(weights / scale), -127, 127).astype(np.int8)
dequant = q.astype(float) * scale
err = dequant - weights
table = pd.DataFrame({'weight': weights.ravel(), 'int8': q.ravel(),
                      'dequantized': dequant.ravel(), 'error': err.ravel()})
print('scale:', scale, ' max abs error:', np.max(np.abs(err)))
table.round(5)

## 執行結果圖

In [ ]:
# 程式 25-04｜投影片 25：3、14、16 頁
x = np.arange(weights.size)
plt.plot(x, weights.ravel(), 'o-', label='float')
plt.plot(x, dequant.ravel(), 'x--', label='dequantized int8')
plt.xlabel('weight index'); plt.ylabel('value'); plt.title('Symmetric int8 quantization')
plt.legend()
savefig('25_quantization_demo')
report('quantization', {'scale': scale, 'max_abs_error': np.max(np.abs(err)), 'codes': q.tolist()})

## 1. 位元格式、BF16 模擬與 loss scaling

BF16 以 FP32 位元四捨六入五成雙後截低 16 位模擬；數值仍存在 FP32 陣列，不宣稱取得 BF16 硬體速度。loss scaling 在梯度產生前放大，最後先 unscale 才 clip。

In [ ]:
# 程式 25-05｜投影片 25：4、5、6、7、8、9、10、11、25、28 頁
formats25=pd.DataFrame({'format':['FP32','FP16','BF16'],'sign':[1,1,1],'exponent':[8,5,8],'fraction':[23,10,7]})
fig,ax=plt.subplots(figsize=(9,3));left25=np.zeros(3)
for name25 in ['sign','exponent','fraction']:
    ax.barh(formats25['format'],formats25[name25],left=left25,label=name25);left25+=formats25[name25].to_numpy()
ax.set_xlabel('bits');ax.legend();savefig('c25_float_bit_layout')
def bf16_sim25(values):
    values=np.asarray(values,dtype=np.float32);u=values.view(np.uint32)
    rounded=u+np.uint32(0x7FFF)+((u>>16)&1)
    return (rounded&np.uint32(0xFFFF0000)).view(np.float32)
values25=np.array([1.0001,1.01,1000.1,1e-8,1e8],np.float32)
with np.errstate(over='ignore'):fp16_25=values25.astype(np.float16).astype(np.float32)
print(pd.DataFrame({'FP32':values25,'FP16':fp16_25,'BF16 simulated':bf16_sim25(values25)}))
assert np.isinf(fp16_25[-1]) and np.isfinite(bf16_sim25(values25)[-1])
small_grad25=np.float32(1e-8);scale25=32768.
underflow25=np.float16(small_grad25);scaled25=np.float16(small_grad25*scale25);recovered25=np.float32(scaled25)/scale25
assert underflow25==0 and recovered25>0
print('Direct FP16 / scaled then FP32 unscale:',underflow25,recovered25)
g25=np.array([3.,4.],np.float32);scaled_g25=g25*scale25;unscaled25=scaled_g25/scale25
clipped25=unscaled25*min(1,1/np.linalg.norm(unscaled25));assert np.isclose(np.linalg.norm(clipped25),1)
nonfinite25=np.array([np.inf,1.]);print('Skip optimizer step for nonfinite gradient:',not np.isfinite(nonfinite25).all())
print('1M weights-only bytes FP32/FP16/int8:',[1_000_000*n for n in [4,2,1]])

## 2. 手算、零張量、對稱／非對稱與每通道

先檢查範圍和 scale，再 round/clip；整數乘法需較寬的累加器。零張量用安全 scale 避免除零。每通道量化各自保存尺度，metadata 也占空間。

In [ ]:
# 程式 25-06｜投影片 25：12、13、14、15、16、18、25、26、28 頁
hand25=np.array([.26,-.44,0.]);codes25=np.round(hand25/.1);restored25=codes25*.1
assert codes25.tolist()==[3,-4,0] and np.allclose(restored25-hand25,[.04,.04,0])
def symmetric25(x,axis=None):
    peak=np.max(np.abs(x),axis=axis,keepdims=True);s=np.where(peak==0,1.,peak/127.)
    q=np.clip(np.round(x/s),-127,127).astype(np.int8);return q,s,q.astype(np.float32)*s
qzero25,szero25,rzero25=symmetric25(np.zeros((2,3)));assert np.isfinite(rzero25).all() and np.all(rzero25==0)
w25=np.array([[.01,.03,-.02],[1.,3.,-2.]],dtype=np.float32)
_,st25,rt25=symmetric25(w25);qc25,sc25,rc25=symmetric25(w25,axis=1)
print('Tensor / channel MSE:',np.mean((rt25-w25)**2),np.mean((rc25-w25)**2),'channel scales:',sc25.ravel())
# uint8 非對稱：校準範圍含0，以可表達零點兼顧正負數
low25,high25=-1.,3.;as25=(high25-low25)/255;zero25=int(np.clip(np.round(-low25/as25),0,255))
data25=np.array([-1.,0.,.26,3.,5.]);aq25=np.clip(np.round(data25/as25)+zero25,0,255).astype(np.uint8);ar25=(aq25.astype(float)-zero25)*as25
print('Asymmetric scale/zero/code/restored:',as25,zero25,aq25,ar25)
# int8 GEMM，int32 累加，最後以 x_scale * w_scale 還原
x25=np.array([[.2,-.7,1.]],np.float32);qx25,sx25,rx25=symmetric25(x25)
acc25=qx25.astype(np.int32)@qc25.astype(np.int32).T;result25=acc25*sx25*sc25.T
assert np.allclose(result25,rx25@rc25.T)
plt.scatter(w25.ravel(),rt25.ravel(),label='per tensor');plt.scatter(w25.ravel(),rc25.ravel(),marker='x',label='per channel');plt.plot([-2,3],[-2,3],'k--');plt.xlabel('original');plt.ylabel('dequantized');plt.legend();savefig('c25_quantization_granularity')

## 3. 校準、動態範圍與 fake quantization

靜態 PTQ 的範圍來自校準資料；測試資料不參與選 scale。動態量化可在推論時依當批範圍取 scale。QAT 前向 fake quant，反向採 straight-through 近似，並非 round 的真正導數。

In [ ]:
# 程式 25-07｜投影片 25：17、18、19、20、21、22、23、24、25、26、27、28 頁
rng25=np.random.default_rng(SEED);calibration25=rng25.normal(size=1000);test25=np.r_[rng25.normal(size=200),8.]
static_scale25=np.max(np.abs(calibration25))/127
static25=np.clip(np.round(test25/static_scale25),-127,127)*static_scale25
_,dynamic_scale25,dynamic25=symmetric25(test25)
print('Calibration/static vs dynamic scale:',static_scale25,dynamic_scale25.ravel())
print('Clipped test fraction:',np.mean(np.abs(test25)>127*static_scale25))
# 玩具 QAT：固定量化步長，目標 .83；只有未超範圍才用STE梯度1
weight25=.1;target25=.83;step25=.1;trace25=[]
for _ in range(60):
    fake25=np.clip(np.round(weight25/step25),-7,7)*step25
    loss25=.5*(fake25-target25)**2;trace25.append([weight25,fake25,loss25])
    ste25=float(-.7<=weight25<=.7);weight25-=.1*(fake25-target25)*ste25
trace25=np.array(trace25)
fig,axes=plt.subplots(1,2,figsize=(11,3.5));axes[0].hist(test25-static25,bins=35,alpha=.6,label='static calibrated');axes[0].hist(test25-dynamic25,bins=35,alpha=.6,label='dynamic batch');axes[0].legend();axes[0].set_xlabel('dequantization error')
axes[1].plot(trace25[:,0],label='FP32 master');axes[1].plot(trace25[:,1],label='fake-quant forward');axes[1].axhline(target25,c='k',ls='--');axes[1].legend();axes[1].set_xlabel('STE update');savefig('c25_ptq_qat')
print('QAT cannot represent target beyond range .7:',trace25[-1])
# 儲存 dtype 與運算 dtype 分開：int8 權重還原成FP32，並未使用低位元運算kernel
metadata25={'stored_dtype':'int8','scale_dtype':'float32','dequantized_compute_dtype':'float32','zero_point':0}
print('Quantized artifact contract:',metadata25,'int8 weight bytes + float scales:',qc25.nbytes+sc25.astype(np.float32).nbytes)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。